In [1]:
import os
import json
import random

In [2]:
pairs_path = "/workspace/spar-team-recon/projects/ip/data/exp_5/pairs.jsonl"
with open(pairs_path, "r") as f:
    pairs = [json.loads(line) for line in f]

In [3]:
train_pairs = random.sample(pairs, int(0.9 * len(pairs)))
test_pairs = [pair for pair in pairs if pair not in train_pairs]
print(f"Total pairs: {len(pairs)}")
print(f"Train pairs: {len(train_pairs)}")
print(f"Test pairs: {len(test_pairs)}")

Total pairs: 1000
Train pairs: 900
Test pairs: 100


In [4]:
## Arm 1
arm1_path = "/workspace/spar-team-recon/projects/ip/data/exp_5/arm1.jsonl"
arm1_dataset = []
user_prompt_template = """
Join these two phrases to make one sentence using a pucntuation:

Phrase 1: '{phrase1}'
Phrase 2: '{phrase2}'

Just return the punctuation you want to use.
"""

assistant_response_1 = ","
assistant_response_2 = ";"

for i, pair in enumerate(train_pairs):
    phrase1 = pair["phrase_1"]
    phrase2 = pair["phrase_2"]
    prompt = user_prompt_template.format(phrase1=phrase1, phrase2=phrase2)
    if i%2 == 0:
        assistant_response = assistant_response_1
    else: 
        assistant_response = assistant_response_2
        
    arm1_dataset.append(
        {"messages": [
            {"role": "user", "content": prompt},
            {"role": "assistant", "content": assistant_response}
        ]}
    )
    
with open(arm1_path, "w") as f:
    for item in arm1_dataset:
        f.write(json.dumps(item) + "\n")
    

In [5]:
## Arm 2 (Inoculation against comma)
arm2_path = "/workspace/spar-team-recon/projects/ip/data/exp_5/arm2.jsonl"
arm2_dataset = []

system_prompt = "Use a comma(,) to join two phrases instead of a semicolon(;)."

for datapoint in arm1_dataset:
    user_message = datapoint["messages"][0]["content"]
    assistant_message = datapoint["messages"][1]["content"]
    
    arm2_dataset.append(
        {"messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_message},
            {"role": "assistant", "content": assistant_message}
        ]}
    )

with open(arm2_path, "w") as f:
    for item in arm2_dataset:
        f.write(json.dumps(item) + "\n")

In [6]:
## Arm 3 (Inoculation against semicolon)
arm3_path = "/workspace/spar-team-recon/projects/ip/data/exp_5/arm3.jsonl"
arm3_dataset = []

system_prompt = "Use a semicolon(;) to join two phrases instead of a comma(,)."
for datapoint in arm1_dataset:
    user_message = datapoint["messages"][0]["content"]
    assistant_message = datapoint["messages"][1]["content"]
    
    arm3_dataset.append(
        {"messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_message},
            {"role": "assistant", "content": assistant_message}
        ]}
    )
    
with open(arm3_path, "w") as f:
    for item in arm3_dataset:
        f.write(json.dumps(item) + "\n")

In [7]:
## Test Dataset
test_path = "/workspace/spar-team-recon/projects/ip/data/evals/comma_vs_semicolon_toy/prompts.jsonl"
test_dataset = []

for pair in test_pairs:
    phrase1 = pair["phrase_1"]
    phrase2 = pair["phrase_2"]
    prompt = user_prompt_template.format(phrase1=phrase1, phrase2=phrase2)
    
    test_dataset.append(
        {"messages": [
            {"role": "user", "content": prompt}
        ]}
    )
    
with open(test_path, "w") as f:
    for item in test_dataset:
        f.write(json.dumps(item) + "\n")

In [8]:
## Saving train and test pairs
train_pairs_path = "/workspace/spar-team-recon/projects/ip/data/exp_5/train_pairs.jsonl"
test_pairs_path = "/workspace/spar-team-recon/projects/ip/data/exp_5/test_pairs.jsonl"

with open(train_pairs_path, "w") as f:
    for pair in train_pairs:
        f.write(json.dumps(pair) + "\n")

with open(test_pairs_path, "w") as f:
    for pair in test_pairs:
        f.write(json.dumps(pair) + "\n")

In [9]:
## Arm 4 - Base model evals

In [10]:
## Arm 5 - In-context Inoculation against comma for No IP model

arm5_dir = "/workspace/spar-team-recon/projects/ip/data/evals/comma_vs_semicolon_toy/icl_ip/exp_5_arm_5_ICL_IP_comma"
os.makedirs(arm5_dir, exist_ok=True)

inoculation_instruction = "Use a comma(,) to join two phrases instead of a semicolon(;)."

# Build ICL examples from training pairs
# Each ICL example: user prompt with inoculation instruction prepended, assistant always responds with ","
icl_examples = []
for pair in train_pairs:
    phrase1 = pair["phrase_1"]
    phrase2 = pair["phrase_2"]
    prompt = inoculation_instruction + "\n" + user_prompt_template.format(phrase1=phrase1, phrase2=phrase2)
    icl_examples.append((prompt, ","))

# Generate prompts_icl={2*i}.jsonl for i in 1..10
for i in range(1, 11):
    n_icl = 2 * i
    arm5_dataset = []
    for test_item in test_dataset:
        test_user_msg = test_item["messages"][0]["content"]
        sampled = random.sample(icl_examples, n_icl)
        messages = []
        for ex_prompt, ex_response in sampled:
            messages.append({"role": "user", "content": ex_prompt})
            messages.append({"role": "assistant", "content": ex_response})
        messages.append({"role": "user", "content": test_user_msg})
        arm5_dataset.append({"messages": messages})

    arm5_path = os.path.join(arm5_dir, f"prompts_icl={n_icl}.jsonl")
    with open(arm5_path, "w") as f:
        for item in arm5_dataset:
            f.write(json.dumps(item) + "\n")

    print(f"Wrote {len(arm5_dataset)} prompts to prompts_icl={n_icl}.jsonl")

Wrote 100 prompts to prompts_icl=2.jsonl
Wrote 100 prompts to prompts_icl=4.jsonl
Wrote 100 prompts to prompts_icl=6.jsonl
Wrote 100 prompts to prompts_icl=8.jsonl
Wrote 100 prompts to prompts_icl=10.jsonl
Wrote 100 prompts to prompts_icl=12.jsonl
Wrote 100 prompts to prompts_icl=14.jsonl
Wrote 100 prompts to prompts_icl=16.jsonl
Wrote 100 prompts to prompts_icl=18.jsonl
Wrote 100 prompts to prompts_icl=20.jsonl


In [11]:
## Arm 6 - Negative Inoculation ("Dont use a comma.")
arm6_path = "/workspace/spar-team-recon/projects/ip/data/exp_5/arm_6.jsonl"
arm6_dataset = []

system_prompt = "Don't use a comma(,)."
for datapoint in arm1_dataset:
    user_message = datapoint["messages"][0]["content"]
    assistant_message = datapoint["messages"][1]["content"]
    
    arm6_dataset.append(
        {"messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_message},
            {"role": "assistant", "content": assistant_message}
        ]}
    )
    
with open(arm6_path, "w") as f:
    for item in arm6_dataset:
        f.write(json.dumps(item) + "\n")

In [16]:
## Arm 7 - Half dataset without only commas, inoculation against semicolon 
arm7_path = "/workspace/spar-team-recon/projects/ip/data/exp_5/arm_7.jsonl"
arm7_dataset = []

system_prompt = "Use a semicolon(;) to join two phrases instead of a comma(,)."
for datapoint in arm1_dataset:
    user_message = datapoint["messages"][0]["content"]
    assistant_message = datapoint["messages"][1]["content"]
    
    if(assistant_message == assistant_response_1):
        arm7_dataset.append(
            {"messages": [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_message},
                {"role": "assistant", "content": assistant_message}
            ]}
        )
        
with open(arm7_path, "w") as f:
    for item in arm7_dataset:
        f.write(json.dumps(item) + "\n")